

# Pseudobulk differential expression: hPGC in vivo vs hPGCLC in vitro

Every in vivo cell came from one set of experiments and every in vitro cell from
another. Condition and batch are therefore **perfectly confounded**. No model can
separate them, because no cell exists that is in vivo and processed the in vitro
way.

1. **Stage matching.** Comparing day 4 hPGCLCs against germ cells from
   6 to 9 PCW puts development in the top hits. `STAGE_MATCHED` restricts
   the in vivo side to early PGCs. I think we discussed to run it both ways:
   the genes that move only in the unmatched version are maturation.
2. **Replication across datasets.** Section 9 exports the set in a form you
   can re-test elsewhere.

### The unit of replication is the sample, not the cell

Pseudobulk sums counts within a sample and treats that as one
observation. Treating individual cells as replicates inflates significance
enormously and is the single most common error in single-cell analysis.
Section 2 stops if you have too few real replicates.
(I don't remember if I explain this one in detail, check this
[https://www.youtube.com/watch?v=04gB2owLKus&list=PLJefJsd1yfhagnkss5B1YCsHaH0GWQfFT&index=8](https://www.youtube.com/watch?v=04gB2owLKus&list=PLJefJsd1yfhagnkss5B1YCsHaH0GWQfFT&index=8)
she is one of my favourite youtubers, I usually recommend her to people that
use Seurat, but the logic behind is the same in scanpy as well.)

*N.B.*: For stage_matched run, min_cells was lowered to 6 (instead of 10), otherwise there would have been no in vivo samples left. With min_cells = 6, we get 1 sample passing.

## 1. Settings

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

# ---- inputs ---------------------------------------------------------------
INVIVO_H5AD  = "path_to_data/invivo_window_female.h5ad" #change to your directory
INVITRO_H5AD = "path_to_data/invitro_integrated_scanpy_joao.h5ad" #change to your directory

INVIVO_CELLTYPE_KEY  = "fine_label"
INVIVO_SAMPLE_KEY    = "donor_id"  
INVIVO_GERM          = "primordial germ cell"  

INVITRO_CELLTYPE_KEY = "cell_type"
INVITRO_SAMPLE_KEY   = "sample"
INVITRO_GERM         = "PGCLC" 

# ---- raw counts -----------------------------------------------------------
# DESeq2 models counts. It must NOT be given log-normalised data. Set to the
# layer holding integers, or None if adata.X itself is raw counts.
COUNTS_LAYER = "counts"  # CHANGE IF NEEDED

# ---- analysis choices -----------------------------------------------------
STAGE_MATCHED   = True  # restrict in vivo to DAZL/DDX4 negative early PGCs
MIN_CELLS       = 6     # a sample needs this many germ cells to contribute
MIN_SAMPLES     = 2      # per condition; below this the analysis is not valid
                         # (usually it would be 3, but I don't remember how many we have)
MIN_COUNTS      = 1e5    # drop pseudobulk samples with fewer total counts
MIN_GENE_SAMPLES = 1     # a gene must be seen in this many samples overall - ignore for now
PADJ_CUTOFF     = 0.05
LFC_CUTOFF      = 1.0    # 2 fold

OUTDIR = "results_pseudobulk"
os.makedirs(OUTDIR, exist_ok=True)

## 2. Load, subset to germ cells, and count replicates

In [ ]:
vivo  = sc.read_h5ad(INVIVO_H5AD)
vitro = sc.read_h5ad(INVITRO_H5AD)

for name, a, ct_key, germ in [("vivo", vivo, INVIVO_CELLTYPE_KEY, INVIVO_GERM),
                              ("vitro", vitro, INVITRO_CELLTYPE_KEY, INVITRO_GERM)]:
    a.obs[ct_key] = a.obs[ct_key].astype(str)
    n = int((a.obs[ct_key] == germ).sum())
    if n == 0:
        raise ValueError(f"{name}: no cells labelled {germ!r} under {ct_key!r}. "
                         f"Present: {sorted(a.obs[ct_key].unique())[:25]}")
    print(f"{name}: {a.n_obs} cells, {n} labelled {germ!r}")

# Optional stage matching. PGC_early is defined as germ cells with no detected
# DAZL or DDX4, which is the closest in vivo counterpart to a day 4 hPGCLC.
if STAGE_MATCHED:
    late = np.zeros(vivo.n_obs, dtype=bool)
    for g in ["DAZL", "DDX4"]:
        if g in vivo.var_names:
            v = vivo[:, g].X
            v = v.toarray().ravel() if hasattr(v, "toarray") else np.asarray(v).ravel()
            late |= (v > 0)
    germ_mask = (vivo.obs[INVIVO_CELLTYPE_KEY] == INVIVO_GERM).values
    keep = germ_mask & ~late
    print(f"\nstage matched: {int(keep.sum())} early of {int(germ_mask.sum())} germ cells")
    germ_vivo = vivo[keep].copy()
else:
    germ_vivo = vivo[vivo.obs[INVIVO_CELLTYPE_KEY] == INVIVO_GERM].copy()

germ_vitro = vitro[vitro.obs[INVITRO_CELLTYPE_KEY] == INVITRO_GERM].copy()

# ---- the check that decides whether this analysis is possible at all -------
n_vivo  = germ_vivo.obs[INVIVO_SAMPLE_KEY].nunique()
n_vitro = germ_vitro.obs[INVITRO_SAMPLE_KEY].nunique()
print(f"\nindependent samples: {n_vivo} in vivo, {n_vitro} in vitro")

if min(n_vivo, n_vitro) < MIN_SAMPLES:
    raise ValueError(
        f"Only {min(n_vivo, n_vitro)} replicates on one side. Pseudobulk DE needs at "
        f"least {MIN_SAMPLES} independent donors or inductions per condition.")

## 3. Confirm we have raw counts

DESeq2 models integer counts and estimates dispersion from them. Handing it
log-normalised values produces confident nonsense rather than an error, so this
check is worth its few lines.
BE CAREFULL HERE!!! We can verify when I am back!

In [ ]:
def get_matrix(adata, layer):
    X = adata.layers[layer] if layer else adata.X
    return X

def check_raw_counts(adata, layer, name):
    X = get_matrix(adata, layer)
    head = X[:200]
    head = head.toarray() if hasattr(head, "toarray") else np.asarray(head)
    is_int = np.allclose(head, np.round(head))
    # max over the WHOLE matrix, not just the head: if the first 200 cells
    # happen to be sparse, a head-only max can fall under 30 and this raises
    # on data that are perfectly good counts.
    mx = float(X.max())
    print(f"{name}: max {mx:.2f}, integer valued {is_int}")
    if not is_int or mx <= 30:
        raise ValueError(
            f"{name} does not look like raw counts (max {mx:.2f}, integer {is_int}). "
            f"Point COUNTS_LAYER at the counts layer, or use adata.raw. Log "
            f"normalised input will not error, it will just give wrong answers.")

check_raw_counts(germ_vivo, COUNTS_LAYER, "vivo")
check_raw_counts(germ_vitro, COUNTS_LAYER, "vitro")

## 4. Match genes across the two objects

This is fine for us, but test anyway. The idea is that if one object uses Ensembl IDs and
the other symbols, convert before this step or the intersection will be empty.

In [ ]:
# Subsetting by gene NAME is ambiguous if a symbol repeats, and the pseudobulk
# step then labels count columns with adata.var_names, so a duplicate would
# silently attach counts to the wrong gene.
for _name, _a in [("vivo", germ_vivo), ("vitro", germ_vitro)]:
    if not _a.var_names.is_unique:
        raise ValueError(
            f"{_name}: {int(_a.var_names.duplicated().sum())} duplicated var_names. "
            f"Run _a.var_names_make_unique() or subset by Ensembl ID first.")

shared_genes = sorted(set(germ_vivo.var_names) & set(germ_vitro.var_names))
print(f"{len(germ_vivo.var_names)} vivo genes, {len(germ_vitro.var_names)} vitro genes, "
      f"{len(shared_genes)} shared")
if len(shared_genes) < 5000:
    raise ValueError("Fewer than 5000 shared genes. The two objects are probably "
                     "using different identifier types (Ensembl versus symbol).")

germ_vivo  = germ_vivo[:, shared_genes].copy()
germ_vitro = germ_vitro[:, shared_genes].copy()

## 5. Pseudobulk

Sum raw counts within each sample. One row per donor or induction, which is the
unit the statistics will treat as independent.

In [ ]:
def pseudobulk(adata, sample_key, condition, layer=COUNTS_LAYER, min_cells=MIN_CELLS):
    X = get_matrix(adata, layer)
    samples, rows, meta = adata.obs[sample_key].astype(str), [], []
    for s in sorted(samples.unique()):
        m = (samples == s).values
        n = int(m.sum())
        if n < min_cells:
            print(f"  dropping {s}: {n} cells, under MIN_CELLS")
            continue
        sub = X[m]
        sub = sub.toarray() if hasattr(sub, "toarray") else np.asarray(sub)
        rows.append(sub.sum(axis=0))
        meta.append({"sample": f"{condition}_{s}", "condition": condition, "n_cells": n})
    if not rows:
        raise ValueError(f"no {condition} sample reached MIN_CELLS")
    counts = pd.DataFrame(np.vstack(rows), columns=adata.var_names,
                          index=[m["sample"] for m in meta])
    return counts, pd.DataFrame(meta).set_index("sample")

cv, mv = pseudobulk(germ_vivo,  INVIVO_SAMPLE_KEY,  "in_vivo")
ct, mt = pseudobulk(germ_vitro, INVITRO_SAMPLE_KEY, "in_vitro")

counts = pd.concat([cv, ct]).round().astype(int)
meta   = pd.concat([mv, mt])
meta["condition"] = pd.Categorical(meta["condition"], categories=["in_vivo", "in_vitro"])

# drop shallow pseudobulk samples, then genes seen in too few samples
deep = counts.sum(axis=1) >= MIN_COUNTS
if (~deep).any():
    print("dropping shallow samples:", list(counts.index[~deep]))
counts, meta = counts[deep], meta[deep]

keep_genes = (counts > 0).sum(axis=0) >= MIN_GENE_SAMPLES
counts = counts.loc[:, keep_genes]
print(f"\n{counts.shape[0]} pseudobulk samples x {counts.shape[1]} genes")
print(meta.groupby("condition", observed=True).size().to_string())

## 6. Look at the samples before testing anything

The samples will separate cleanly by condition on PC1. That is both the result
and the issue I mentioned in the beginning.

What to actually look for here is whether any sample sits apart from its own
group. That would be a technical outlier worth removing before testing.

In [ ]:
from sklearn.decomposition import PCA

cpm = counts.div(counts.sum(axis=1), axis=0) * 1e6
logcpm = np.log1p(cpm)
pcs = PCA(n_components=2).fit(logcpm.values)
xy = pcs.transform(logcpm.values)

fig, ax = plt.subplots(figsize=(6, 5))
for cond, colour in [("in_vivo", "#2166ac"), ("in_vitro", "#b2182b")]:
    m = (meta["condition"] == cond).values
    ax.scatter(xy[m, 0], xy[m, 1], s=70, c=colour, label=cond, edgecolor="white")
for i, s in enumerate(counts.index):
    ax.annotate(s, (xy[i, 0], xy[i, 1]), fontsize=7, xytext=(4, 4),
                textcoords="offset points")
ax.set_xlabel(f"PC1 ({pcs.explained_variance_ratio_[0]:.0%})")
ax.set_ylabel(f"PC2 ({pcs.explained_variance_ratio_[1]:.0%})")
ax.set_title("Pseudobulk samples (condition and batch are confounded)")
ax.legend(frameon=False)
plt.savefig(os.path.join(OUTDIR, "pseudobulk_pca.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Differential expression

The contrast is in vitro relative to in vivo, so a **negative** log2 fold change
means lower in hPGCLCs, which is the direction of interest: present in the
embryo, missing in the dish.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

try:                                     # pydeseq2 >= 0.5
    dds = DeseqDataSet(counts=counts, metadata=meta, design="~condition",
                       refit_cooks=True)
except TypeError:                        # older releases
    dds = DeseqDataSet(counts=counts, metadata=meta, design_factors="condition",
                       refit_cooks=True)
dds.deseq2()

stat = DeseqStats(dds, contrast=["condition", "in_vitro", "in_vivo"])
stat.summary()
res = stat.results_df.copy()
res["gene"] = res.index
res = res.dropna(subset=["padj"]).sort_values("padj")

res["direction"] = np.where(
    (res.padj < PADJ_CUTOFF) & (res.log2FoldChange <= -LFC_CUTOFF), "down in hPGCLC",
    np.where((res.padj < PADJ_CUTOFF) & (res.log2FoldChange >= LFC_CUTOFF),
             "up in hPGCLC", "ns"))

print(res.direction.value_counts().to_string())
res.to_csv(os.path.join(OUTDIR, f"de_{'stagematched' if STAGE_MATCHED else 'allstages'}.csv"),
           index=False)
res.head(20)

## 8. Volcano

In [ ]:
COL = {"down in hPGCLC": "#2166ac", "up in hPGCLC": "#b2182b", "ns": "#d9d9d9"}

fig, ax = plt.subplots(figsize=(7, 6))
y = -np.log10(res.padj.clip(lower=1e-300))
for lab in ["ns", "up in hPGCLC", "down in hPGCLC"]:
    m = (res.direction == lab).values
    ax.scatter(res.log2FoldChange[m], y[m], s=8, c=COL[lab], label=lab,
               alpha=0.6 if lab == "ns" else 0.85, linewidths=0)

for lab in ["down in hPGCLC", "up in hPGCLC"]:
    for _, r in res[res.direction == lab].head(12).iterrows():
        ax.annotate(r.gene, (r.log2FoldChange, -np.log10(max(r.padj, 1e-300))),
                    fontsize=7, xytext=(3, 3), textcoords="offset points")

ax.axhline(-np.log10(PADJ_CUTOFF), ls="--", lw=0.8, c="grey")
ax.axvline(-LFC_CUTOFF, ls="--", lw=0.8, c="grey")
ax.axvline(LFC_CUTOFF, ls="--", lw=0.8, c="grey")
ax.set_xlabel("log2 fold change, hPGCLC relative to hPGC")
ax.set_ylabel("-log10 adjusted p")
ax.set_title("hPGCLC versus hPGC" + (" (stage matched)" if STAGE_MATCHED else ""))
ax.legend(frameon=False, markerscale=2, fontsize=8)
plt.savefig(os.path.join(OUTDIR, "volcano.png"), dpi=150, bbox_inches="tight")
plt.show()

## 9. Export the gene set for NicheNet

This also connects to the ligand receptor work. NicheNet is the alternative tool
I mentioned in the first meeting (Niky from MRC used it, and she said she got good
results), it takes a gene set of interest and ranks candidate ligands by how well
their known downstream targets explain it. The genes **down** in hPGCLCs are that
set: the programme the culture fails to switch on.

The background must be genes that were actually testable here, not the whole
genome, or the enrichment will be inflated.

Since you are doing this DA, might as well keep them for future use.

In [ ]:
down = res[res.direction == "down in hPGCLC"].sort_values("log2FoldChange")
background = res.gene.tolist()

tag = "stagematched" if STAGE_MATCHED else "allstages"
down.to_csv(os.path.join(OUTDIR, f"geneset_down_in_hpgclc_{tag}.csv"), index=False)
pd.Series(background, name="gene").to_csv(
    os.path.join(OUTDIR, f"background_{tag}.csv"), index=False)

print(f"{len(down)} genes down in hPGCLC, background of {len(background)}")
print("\nTop 30 by fold change:")
print(down.head(30)[["gene", "log2FoldChange", "padj"]].to_string(index=False))

## 10. Run it both ways

Do not forget to `STAGE_MATCHED = True`, re-run, and compare the two gene lists.

In [ ]:
from scipy.stats import hypergeom
from matplotlib_venn import venn2

# ---- 1. Load both gene sets + their backgrounds ----
down_all    = pd.read_csv("results_pseudobulk/geneset_down_in_hpgclc_allstages.csv")
down_matched  = pd.read_csv("results_pseudobulk/geneset_down_in_hpgclc_stagematched.csv")
bg_all      = set(pd.read_csv("results_pseudobulk/background_allstages.csv")["gene"])
bg_matched    = set(pd.read_csv("results_pseudobulk/background_stagematched.csv")["gene"])

genes_all   = set(down_all["gene"])
genes_matched = set(down_matched["gene"])

# ---- 2. Basic overlap ----
shared      = genes_all & genes_matched
only_all    = genes_all - genes_matched
only_matched  = genes_matched - genes_all
union       = genes_all | genes_matched
jaccard     = len(shared) / len(union) if union else np.nan

print(f"All-stages down:    {len(genes_all)}")
print(f"Stage-matched down: {len(genes_matched)}")
print(f"Shared:             {len(shared)}")
print(f"Only all-stages:    {len(only_all)}")
print(f"Only stage-matched: {len(only_matched)}")
print(f"Jaccard index:      {jaccard:.3f}")

# ---- 3. Is the overlap more than expected by chance? ----
# Use the intersection of the two backgrounds as the common universe —
# a gene can only be a valid "hit" in both analyses if it was testable in both.
common_background = bg_all & bg_matched
N = len(common_background)                      # universe size
K = len(genes_all & common_background)           # "successes" in population (all-stages hits)
n = len(genes_matched & common_background)         # draws (stage-matched hits)
k = len(shared & common_background)              # observed overlap

pval = hypergeom.sf(k - 1, N, K, n)              # P(overlap >= k)
print(f"\nCommon background size: {N}")
print(f"Hypergeometric overlap test: k={k}, expected={K*n/N:.1f}, p={pval:.3e}")

# ---- 4. Direction/magnitude concordance for shared genes ----
merged = down_all.merge(
    down_matched, on="gene", suffixes=("_allstages", "_stagematched")
)
corr = merged["log2FoldChange_allstages"].corr(merged["log2FoldChange_stagematched"])
print(f"\nlog2FC correlation for shared genes (Pearson r): {corr:.3f}")

# flag any shared genes with discordant direction (shouldn't happen if both
# lists are already filtered to "down", but useful if you re-run this on
# unfiltered up+down lists)
discordant = merged[
    np.sign(merged["log2FoldChange_allstages"]) != np.sign(merged["log2FoldChange_stagematched"])
]
if len(discordant):
    print(f"WARNING: {len(discordant)} shared genes have discordant fold-change direction")

# ---- 5. Save the comparison tables ----
merged.to_csv(os.path.join(OUTDIR, "shared_down_genes_comparison.csv"), index=False)

# ---- 6. Venn diagram ----
fig, ax = plt.subplots(figsize=(5, 5))
v = venn2(
    [genes_all, genes_matched],
    set_labels=("All stages", "Stage matched"),
    set_colors=("#2a78d6", "#eb6834"),
    alpha=0.85,
    ax=ax,
)
for text in v.set_labels:
    if text: text.set_color("#0b0b0b")
for text in v.subset_labels:
    if text: text.set_color("#0b0b0b")
ax.set_title("Genes down in hPGCLC", color="#0b0b0b")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "venn_down_in_hpgclc.png"), dpi=200)
plt.show()

## Readout of genes downregulated in hPGCLCs on the volcano plot

In [ ]:
down_genes = res[res.direction == "down in hPGCLC"].copy()

# Combined rank: most extreme on BOTH axes (far left AND high up)
down_genes["lfc_rank"]      = down_genes["log2FoldChange"].rank()   # ascending → most negative LFC = rank 1
down_genes["padj_rank"]     = down_genes["padj"].rank()             # ascending → smallest padj = rank 1
down_genes["combined_rank"] = down_genes["lfc_rank"] + down_genes["padj_rank"]

top_genes = down_genes.sort_values("combined_rank").head(30)
print(top_genes[["gene", "log2FoldChange", "padj"]].to_string(index=False))

## Visual of down genes (most significant and largest magnitude change)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

# validated sequential blue ramp (light -> dark = low -> high significance)
blue_ramp = LinearSegmentedColormap.from_list(
    "blue_seq",
    ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#0d366b"],
)

top = top_genes.sort_values("log2FoldChange").iloc[::-1]  # reverse for horizontal bar order
neglog10p = -np.log10(top["padj"].clip(lower=1e-300))

fig, ax = plt.subplots(figsize=(6, 8))
bar_colors = blue_ramp((neglog10p - neglog10p.min()) / (neglog10p.max() - neglog10p.min()))
ax.barh(top["gene"], top["log2FoldChange"], color=bar_colors, height=0.65)

ax.set_xlabel("log2FC (down in hPGCLC)", color="#0b0b0b")
ax.set_title("Top 30 genes: most extreme on fold-change AND significance",
             color="#0b0b0b", fontsize=11)
ax.tick_params(colors="#52514e")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#c3c2b7")
ax.spines["bottom"].set_color("#c3c2b7")
ax.axvline(0, color="#c3c2b7", linewidth=1)

# colorbar as the legend for the significance channel
sm = ScalarMappable(cmap=blue_ramp)
sm.set_array(neglog10p)
cbar = fig.colorbar(sm, ax=ax, fraction=0.04, pad=0.02)
cbar.set_label("-log10 adjusted p", color="#0b0b0b", fontsize=9)
cbar.ax.tick_params(colors="#52514e", labelsize=7)

plt.tight_layout()
#plt.savefig("top_genes_combined.pdf", bbox_inches="tight")
#plt.savefig("top_genes_combined.png", dpi=300, bbox_inches="tight")
plt.show()